# Instacart Grocery Recommendations 
This notebook contains the code for our exam project in 02807 Computational Tools for Data Science.  

The goal of our project is to implement and evaluate multiple recommenders for groceries. We are working with the [Instacart Online Grocery Basket Analysis Dataset](https://www.kaggle.com/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset/data?select=order_products__prior.csv), which includes approximately 50,000 products, 200,000 users, and 3.4 million orders. 

We use the following algorithms for our recommenders:
- KMeans, clustering
- Apriori, market basket analysis
- Collaborative Filtering (CF), recommender system

Our project introduces the following recommenders:
- Baseline: Top-n recommender
- Apriori Recommender
- CF on full user-item matrix (CF_Full)
- CF on user-aisle matrix (CF_A)
- CF with user clusters and all items (CF_C)
- CF with user clusters and aisles (CF_AC)

Contributors:
- Andreas Kruse Svenningsen (s253844)
- Frederik Winther Bæk (s214618)
- Georgios Loulakis (s252920)
- Sebastian Nygaard Wærling (s254120)

# Environment setup

To run this notebook, a python environment must be setup first. We use python 3.10 for this project.

We provide a short concise guide for setting up the environtment. 
To setup the environment we will use `uv`, a fast python package manager and python version manager written in rust. Information about `uv` can be found here on their [Github](https://github.com/astral-sh/uv) or their [Docs](https://docs.astral.sh/uv/).

Start by cloning our repository [https://github.com/krusand/02807-CT-Project](https://github.com/krusand/02807-CT-Project). 

Using HTTPS: `git clone https://github.com/krusand/02807-CT-Project.git`

Change into the clone directory: `cd 02807-CT-Project`





## MacOS / Linux

On MacOS / Linux, `uv` can be installed using either

brew (Recommended for MacOS): `brew install uv` \
curl: `curl -LsSf https://astral.sh/uv/install.sh | sh` \
wget: `wget -qO- https://astral.sh/uv/install.sh | sh`

If prompted, run `source $HOME/.local/bin/env`

Afterwards, install the specific python version.\
In this project we use python 3.10

Install using
`uv python install 3.10`

Sync dependencies to our environment using
`uv sync`

**The environment is now setup**

To run scripts from the command line
`uv run {script_name}.py`

In jupyter notebooks, use the environment in
`.venv/bin/python`

Activate the environment using CLI
`source .venv/bin/activate`



## Windows

On Windows, `uv` can be installed using

`powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`

Afterwards, we install the specific python version.\
In this project we use python 3.10

Install using
`uv python install 3.10`

Sync dependencies to our environment using
`uv sync`

**The environment is now setup**

To run scripts from the command line
`uv run {script_name}.py`

In jupyter notebooks, use the environment in
`.venv/bin/python`

Activate the environment using CLI
`.venv/Scripts/activate`


# Code

In [ ]:
# imports
import os
import pickle as pkl
import sys 

import kagglehub
import numpy as np
import pandas as pd
from pandas.api.types import CategoricalDtype
import scipy.sparse as sparse
import shutil
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".")))
from config import *

/Users/krusand/Documents/GitHub/02807-CT-Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The following code blocks follow the execution order of scripts specified in the `pipeline.py` script. Some of the recommenders require an substantial amount of memory (RAM) and have been executed using the DTU HPC. We have decided to exclude those part from this notebook. It will be stated when code has been excluded.  

## 1. Download Dataset
The following functions are used to download the Instacart dataset from Kaggle.

In [1]:
def download_dataset() -> str:
    # Download latest version
    path = kagglehub.dataset_download("yasserh/instacart-online-grocery-basket-analysis-dataset")

    print("Path to dataset files:", path)
    return path

def move_dataset_from_cache_to_folder(path_to_cache: str, path_to_folder: str) -> None:
    shutil.copytree(path_to_cache, path_to_folder, dirs_exist_ok=True)
    shutil.rmtree(path_to_folder / "data", ignore_errors=True)

def convert_to_parquet() -> None:

    for file in tqdm(os.listdir(DATA_RAW_DIR)):
        file_name, file_extension = file.split(".")
        file_extension = "."+(file_extension)
        pd.read_csv(DATA_RAW_DIR / (file_name + file_extension)).to_parquet(DATA_CLEANED_DIR / (file_name + ".pq"))

Now we can use the functions to download the dataset:

In [ ]:
path_to_cache = download_dataset()
move_dataset_from_cache_to_folder(path_to_cache=path_to_cache, path_to_folder=DATA_RAW_DIR)
convert_to_parquet()

## 2. Data Split
The following code block saves the raw data to parquet files and divides the orders into a train, validation, and test set. The three sets are also saved as parquet files named `order_products__train`.  

In [ ]:
# load data
orders_df = pd.read_csv(ORDERS_PATH_CSV)
op_prior = pd.read_csv(ORDER_PRODUCTS__PRIOR_PATH_CSV)
op_train = pd.read_csv(ORDER_PRODUCTS__TRAIN_PATH_CSV)

# remove test orders
orders_df = orders_df[orders_df["eval_set"] != "test"]

# sorting to ensure correct ordering
orders_df = orders_df.sort_values(["user_id", "order_number"])

# helper column counting number of orders per user
orders_df["n_orders"] = orders_df.groupby("user_id")["order_number"].transform("max")

# assign split labels (order 1,...,n-2: train, order n-1: val, order n: test)
orders_df["eval_set_new"] = "train"
orders_df.loc[orders_df["order_number"] == orders_df["n_orders"], "eval_set_new"] = "test"
orders_df.loc[orders_df["order_number"] == orders_df["n_orders"] - 1, "eval_set_new"] = "val"

# drop n_orders and make eval_set_new the new eval_set column
orders_df["eval_set"] = orders_df["eval_set_new"]
orders_df = orders_df.drop(columns=["eval_set_new", "n_orders"])

# save orders_df to parquet
orders_df.to_parquet(ORDERS_PATH)

# concatenate order_products data
op_combined = pd.concat([op_prior, op_train])

# order_ids in each split
train_orders = orders_df[orders_df["eval_set"]=="train"]["order_id"]
val_orders = orders_df[orders_df["eval_set"]=="val"]["order_id"]
test_orders = orders_df[orders_df["eval_set"]=="test"]["order_id"]

# order products for each split
op_train_new = op_combined[op_combined["order_id"].isin(train_orders)]
op_val_new = op_combined[op_combined["order_id"].isin(val_orders)]
op_test_new = op_combined[op_combined["order_id"].isin(test_orders)]

# saving to parquet
op_train_new.to_parquet(ORDER_PRODUCTS__TRAIN_PATH)
op_val_new.to_parquet(ORDER_PRODUCTS__VAL_PATH)
op_test_new.to_parquet(ORDER_PRODUCTS__TEST_PATH)

## 3. Calculate Rating

The following code block contains functions used to calculate ratings.

In [ ]:
def calculate_user_product_frequency(merged_df: pd.DataFrame) -> None: 
    logging.info("")
    bui = (
        merged_df.groupby(['user_id', 'product_id'])['order_id']
        .nunique()
        .reset_index(name='Bui')
    )

    bu = (
        merged_df.groupby(['user_id'])['order_id']
        .nunique()
        .reset_index(name='Bu')
    )

    freq_df = pd.merge(bui, bu, on='user_id', how='left')
    freq_df['freq_ui'] = freq_df['Bui'] / freq_df['Bu']
    file_path = DATA_PREPROCESSED_DIR / "user_product_frequency.pq"
    freq_df.to_parquet(file_path, index=False)
    logging.info(f"Saved user_product_frequency to {file_path}")



def calculate_user_product_recency(merged_df: pd.DataFrame, lam: float = 0.0015) -> None:
    logging.info("")
    # replace NaN for first orders
    merged_df['days_since_prior_order'] = merged_df['days_since_prior_order'].fillna(0)

    # sort to compute cumulative time for each user
    merged_df = merged_df.sort_values(['user_id', 'order_number'])

    # cumulative days since first order
    merged_df['cum_days'] = merged_df.groupby('user_id')['days_since_prior_order'].cumsum()

    # total days each user is active
    merged_df['total_days'] = merged_df.groupby('user_id')['days_since_prior_order'].transform('sum')

    # age_days = time until last order
    merged_df['age_days'] = merged_df['total_days'] - merged_df['cum_days']

    # exponential weight based on recency
    merged_df['weight'] = np.exp(-lam * merged_df['age_days'])

    
    freq = (
        merged_df.groupby(['user_id', 'product_id'])['weight']
        .sum()
        .reset_index(name='score')
    )

    file_path = DATA_PREPROCESSED_DIR / "user_product_recency.pq"

    freq.to_parquet(file_path, index=False)
    logging.info(f"Saved user_product_recency to {file_path}")

    freq['recency_score_min_maxed'] = (
        (freq['score'] - freq['score'].min()) /
        (freq['score'].max() - freq['score'].min())
    )

    file_path = DATA_PREPROCESSED_DIR / "user_product_recency_min_max_scaled.pq"

    freq.to_parquet(file_path, index=False)
    logging.info(f"Saved user_product_recency min_max_scaled to {file_path}")



def calculate_tf_idf(merged_df: pd.DataFrame, orders: pd.DataFrame) -> None:
    logging.info("")
    tf = (merged_df
          .groupby(['user_id', 'product_id'])
          .size()
          .reset_index(name='purchase_count')
        )
    
    total_products_per_user = (merged_df
                               .groupby('user_id')
                               .size()
                               .reset_index(name='total_products')
                               )

    tf = pd.merge(tf, total_products_per_user, on='user_id', how='left')
    
    tf['tf'] = tf['purchase_count'] / tf['total_products']

    # IDF = log(total users / users who bought this product)
    total_users = orders['user_id'].nunique()
    users_per_product = merged_df.groupby('product_id')['user_id'].nunique().reset_index(name='users_who_bought')

    # Avoid division by zero
    users_per_product['users_who_bought'] = users_per_product['users_who_bought'] + 1e-5
    users_per_product['idf'] = np.log(total_users / users_per_product['users_who_bought'])

    tf_idf = pd.merge(
        tf[['user_id', 'product_id', 'tf']],
        users_per_product[['product_id', 'idf']],
        on='product_id',
        how='left'
    )

    tf_idf['tfidf_score'] = tf_idf['tf'] * tf_idf['idf']


    final_tfidf_scores = tf_idf[['user_id', 'product_id', 'tfidf_score']]
    
    file_path = DATA_PREPROCESSED_DIR / "user_product_tfidf.pq"

    final_tfidf_scores.to_parquet(file_path, index=False)
    logging.info(f"Saved user_product_tfidf to {file_path}")

    # min-max scaling
    final_tfidf_scores['tfidf_score_min_maxed'] = (
        (final_tfidf_scores['tfidf_score'] - final_tfidf_scores['tfidf_score'].min()) /
        (final_tfidf_scores['tfidf_score'].max() - final_tfidf_scores['tfidf_score'].min())
    )

    file_path = DATA_PREPROCESSED_DIR / "user_product_tfidf_min_max_scaled.pq"

    final_tfidf_scores.to_parquet(file_path, index=False)
    logging.info(f"Saved user_product_tfidf min_max_scaled to {file_path}")

def combine_ratings(mode: str) -> None:
    logging.info("")
    frequency = pd.read_parquet(DATA_PREPROCESSED_DIR / "user_product_frequency.pq")
    recency = pd.read_parquet(DATA_PREPROCESSED_DIR / "user_product_recency_min_max_scaled.pq")
    tfidf = pd.read_parquet(DATA_PREPROCESSED_DIR / "user_product_tfidf_min_max_scaled.pq")

    merged = (
        frequency.merge(recency, on=['user_id', 'product_id'], how='inner')
                 .merge(tfidf, on=['user_id', 'product_id'], how='inner')
    )

    w_freq, w_rec, w_tfidf = ((1/3), (1/3), (1/3))

    merged['ranke_ui'] = (
        w_freq * merged['freq_ui'] +
        w_rec * merged['recency_score_min_maxed'] +
        w_tfidf * merged['tfidf_score_min_maxed']
    )

    final_ratings = merged[['user_id', 'product_id', 'ranke_ui']]
    final_ratings.columns = ["user", "item", 'rating']

    filename_parquet = f"{mode}_ratings_w_freq-{w_freq:.2f}_w_rec-{w_rec:.2f}_w_tfidf-{w_tfidf:.2f}.pq"
    file_path = DATA_PREPROCESSED_DIR / filename_parquet
    final_ratings.to_parquet(file_path, index=False)
    logging.info(f"Saved ratings to {file_path}")

def save_sparse_matrix() -> None:
    logging.info("")
    ratings_long = pd.read_parquet(DATA_PREPROCESSED_DIR / "ratings_w_freq-0.33_w_rec-0.33_w_tfidf-0.33.pq")

    users = ratings_long["user"].unique()
    products = ratings_long["item"].unique()
    shape = (len(users), len(products))

    # Create indices for users and movies
    user_cat = CategoricalDtype(categories=sorted(users), ordered=True)
    product_cat = CategoricalDtype(categories=sorted(products), ordered=True)
    user_index = ratings_long["user"].astype(user_cat).cat.codes
    product_index = ratings_long["item"].astype(product_cat).cat.codes

    # Conversion via COO matrix
    coo = sparse.coo_matrix((ratings_long["rating"], (user_index, product_index)), shape=shape)
    ratings_matrix = coo.tocsr()

    with open(DATA_PREPROCESSED_DIR / "ratings_csr_matrix.pkl", 'wb') as fp:
        pkl.dump(ratings_matrix, file=fp)

def save_unique_users() -> None:
    logging.info("")
    ratings_long = pd.read_parquet(DATA_PREPROCESSED_DIR / "train_ratings_w_freq-0.33_w_rec-0.33_w_tfidf-0.33.pq")

    users = ratings_long["user"].drop_duplicates().to_frame().reset_index()

    file_path = DATA_PREPROCESSED_DIR / "unique_users.pq"
    users.to_parquet(file_path, index=False)
    logging.info(f"Saved ratings to {file_path}")

def save_unique_products() -> None:
    logging.info("")
    ratings_long = pd.read_parquet(DATA_PREPROCESSED_DIR / "train_ratings_w_freq-0.33_w_rec-0.33_w_tfidf-0.33.pq")

    products = ratings_long["item"].drop_duplicates().to_frame().reset_index()

    file_path = DATA_PREPROCESSED_DIR / "unique_products.pq"
    products.to_parquet(file_path, index=False)
    logging.info(f"Saved ratings to {file_path}")

def save_aisle_ratings(mode: str) -> None:
    logging.info("")
    # loading products and ratings dataframes
    products_df = pd.read_csv(PRODUCTS_PATH_CSV)
    ratings_df = pd.read_parquet(f"{DATA_PREPROCESSED_DIR}/{mode}_ratings_w_freq-0.33_w_rec-0.33_w_tfidf-0.33.pq")

    # left joining the products dataframe onto the ratings dataframe
    joined_df = ratings_df.merge(products_df, how="left", left_on="item", right_on="product_id")

    # grouping by user and aisle_id and computing the average rating per (user, aisle_id)
    grp_df = joined_df.groupby(["user", "aisle_id"])["rating"].mean().reset_index()
    grp_df = grp_df.rename(columns={"aisle_id": "item"})

    # saving the aisle ratings
    file_path = f"{DATA_PREPROCESSED_DIR}/{mode}_aisle_ratings_w_freq-0.33_w_rec-0.33_w_tfidf-0.33.pq"
    grp_df.to_parquet(file_path, index=False)
    logging.info(f"Saved ratings to {file_path}")

def save_aisle_top_products() -> None:
    logging.info("")
    # loading order products train and products dataframes
    op_train = pd.read_parquet(ORDER_PRODUCTS__TRAIN_PATH)
    products_df = pd.read_csv(PRODUCTS_PATH_CSV)

    # left joining products_df onto op_train
    merged_df = op_train.merge(products_df, how="left", on="product_id")

    # counting occurrence for each product per aisle
    agg_df = (merged_df
              .groupby(["product_id", "aisle_id"])
              .size()
              .reset_index(name="count")
              .sort_values(['aisle_id', 'count'], ascending=[True, False])
              )

    # saving dictionary to parquet
    file_path = AISLE_TOP_PRODUCTS_PATH # Fixed
    agg_df.to_parquet(file_path, index=False)
    logging.info(f"Saved aisle top products to {file_path}")


And this code block actually calculates the ratings:

In [ ]:
path_dict = {"train": ORDER_PRODUCTS__TRAIN_PATH,
             "val": ORDER_PRODUCTS__VAL_PATH,
             "test": ORDER_PRODUCTS__TEST_PATH,
             }

orders = pd.read_parquet(ORDERS_PATH)

for mode, path in path_dict.items():
    order_products = pd.read_parquet(path)
    
    merged_df = pd.merge(
        order_products,
        orders,
        on="order_id",
        how="inner"
    )

    calculate_user_product_frequency(merged_df=merged_df.copy())
    calculate_user_product_recency(merged_df=merged_df.copy(), lam=0.0015)
    calculate_tf_idf(merged_df=merged_df.copy(), orders=orders.copy())
    combine_ratings(mode=mode)

    if mode == "train":
        save_unique_users()
        save_unique_products()
        save_aisle_top_products()

    save_aisle_ratings(mode=mode)

## 4. Clustering

## 5. Baseline Recommender

## 6. Collaborative Filtering

## 7. Apriori